# Multiple-choice accuracy through depth

Compare trained probes and logit-lens readouts on ARC-Easy, ARC-Challenge,
HellaSwag, and WinoGrande. The shared paper groups determine the recurrent
models and their matched feedforward controls. Each recurrent model has a
benchmark grid; separate ARC-Easy probe and logit-lens summaries use the main
models, and a two-row coda-lens summary uses all eight model panels. A second
logit-lens summary reads Parcae through its learned core-output projection C.

Both modes use the same model–recurrence pairs and produce the same figures.
`mode="quick"` uses eight training examples and one test example per dataset and
saves only in the run directory. `mode="full"` uses up to 5,000 training examples
and 256 test examples and also refreshes the paper figures. Sample overrides
change the amount of data; only the mode controls paper exports.

In [ ]:
mode = "quick"
# Shared main/appendix groups, including Parcae 770M and its matched control.
recurrences = 32
ouro_recurrences = 8
datasets = ["arc_easy", "arc_challenge", "hellaswag", "winogrande"]
limit = None  # Test examples: 1 (quick), 256 (full); 0 uses the whole split.
max_train = None  # Training examples: 8 (quick), 5000 (full); 0 uses the whole split.
probe_batch_size = 4  # Default questions per probe-training update.
probe_batch_size_overrides = {
    "raven": 1,
    "retrofitted_olmo": 8,
    "parcae_140m": 8,
    "parcae_370m": 8,
    "llama_1b": 8,
    "tinyllama": 8,
    "olmo2": 8,
    "parcae_140m_rec1": 16,
    "parcae_370m_rec1": 16,
}
probe_learning_rate = 5e-4  # Adam, one pass over the selected training examples.
seed = 42
device = "auto"
dtype = "auto"
output_dir = None

cache_dir = None  # Defaults to LOOPED_INTERNALS_CACHE_DIR or /data-gauss/hugbla/looped-internals-cache.
cache_mode = None  # Environment default; "reuse", "require" (no inference), or "off".
analysis_version = 1  # Bump when this notebook's scientific reduction changes.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

In [ ]:
import csv
from dataclasses import asdict
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np

from experiments.cache import ExperimentCache
from experiments.mcq_support import collect_benchmark
from experiments.paper_models import PAPER_MODEL_GROUPS, paper_model_pairs
from experiments.support import (
    BASELINE_FOR,
    COLORMAPS,
    MODEL_LABELS,
    ModelSession,
    add_depth_colorbar,
    add_prelude_coda_connectors,
    add_prelude_coda_legend,
    configure_experiment,
    export_paper_figures,
    get_figsize,
    model_subplots,
    sample_indices,
    save_figure,
    save_results,
)
from looped_internals.evaluation import (
    load_mcq_dataset,
)
from looped_internals.evaluation.probes import save_probes

plt.style.use(ROOT / "plots" / "matlab.mplstyle")

# Resolve shared model groups after parameter overrides.
recurrent_pairs, figure_pairs = paper_model_pairs(
    recurrences,
    [
        (name, ouro_recurrences if name == "ouro_1_4b" else recurrences)
        for name in dict.fromkeys(PAPER_MODEL_GROUPS["all"])
    ],
)
feedforward_pairs = list(
    dict.fromkeys((BASELINE_FOR[name], 1) for name, _ in recurrent_pairs)
)
feedforward_figure_pairs = [
    pair
    for name in (
        "llama_1b",
        "olmo2",
        "tinyllama",
        "parcae_370m_rec1",
        "parcae_770m_rec1",
    )
    for pair in feedforward_pairs
    if pair[0] == name
]
model_recurrences = recurrent_pairs + feedforward_pairs
# Ouro has no coda: keep its panel but omit this extra readout.
coda_model_recurrences = [pair for pair in recurrent_pairs if pair[0] != "ouro_1_4b"]
# Parcae panels of the main logit-lens summary, with their controls, are also read through C.
projected_pairs = [
    pair
    for name, steps in figure_pairs["main"]
    for pair in ((name, steps), (BASELINE_FOR[name], 1))
    if pair[0].startswith("parcae_")
]
sample_limit = limit if limit is not None else (1 if mode == "quick" else 256)
train_limit = max_train if max_train is not None else (8 if mode == "quick" else 5000)
if type(train_limit) is not int or train_limit < 0:
    raise ValueError("max_train must be a nonnegative integer or None")
if type(probe_batch_size) is not int or probe_batch_size < 1:
    raise ValueError("probe_batch_size must be a positive integer")
if any(
    type(value) is not int or value < 1 for value in probe_batch_size_overrides.values()
):
    raise ValueError("probe batch-size overrides must be positive integers")
probe_batch_sizes = {
    name: probe_batch_size_overrides.get(name, probe_batch_size)
    for name, _ in model_recurrences
}

configs = {
    (name, recurrences): configure_experiment(
        experiment="multiple_choice",
        mode=mode,
        models=[name],
        num_steps=recurrences,
        limit=sample_limit,
        seed=seed,
        device=device,
        dtype=dtype,
        output_dir=output_dir,
    )
    for name, recurrences in model_recurrences
}
config = next(iter(configs.values()))

METRICS = {
    "probe_accuracy": "Linear probe accuracy",
    "logitlens_accuracy": "Logit lens accuracy",
}
DATASET_LABELS = {
    "arc_easy": "ARC-Easy",
    "arc_challenge": "ARC-Challenge",
    "hellaswag": "HellaSwag",
    "winogrande": "WinoGrande",
}
PAPER_NAMES = {
    "raven": "huginn",
    "ouro_1_4b": "ouro",
    "retrofitted_llama": "retrofitted-llama",
    "retrofitted_olmo": "retrofitted-olmo",
    "retrofitted_tinyllama": "retrofitted-tinyllama",
    **{
        name: name.replace("_", "-")
        for name, _ in recurrent_pairs
        if name.startswith("parcae_")
    },
}

result_cache = ExperimentCache(cache_dir, cache_mode)

In [ ]:
def accuracy_curves(result, metric, *, by_recurrence=False):
    """Use execution depth for baselines and physical layer depth for recurrences."""
    blocks = result.per_virtual_layer_metrics
    positions = np.array(
        [
            block.layer_index if by_recurrence else block.virtual_layer_index
            for block in blocks
        ],
        dtype=float,
    )
    positions = 100 * positions / max(positions.max(), 1)
    groups = (
        result.get_metric(metric, group_by="recurrence")
        if by_recurrence
        else {"all": result.get_metric(metric)}
    )
    # Prelude, recurrences, and coda partition the recorded execution in order.
    curves = {}
    start = 0
    for group, accuracy in groups.items():
        end = start + len(accuracy)
        curves[group] = (positions[start:end], accuracy.numpy())
        start = end
    return curves

## Train and evaluate

Select each dataset once, using the same seeded shuffle for every model. Train on
the training split; evaluate ARC on its test split and HellaSwag/WinoGrande on
validation. ARC keeps only questions with four valid choices, so saved indices
refer to the filtered examples. Training and test selections are saved separately.

Load each required model–recurrence pair once and process all four datasets.
Train independent probes for every dataset and virtual layer, with the model
weights frozen, then evaluate both readouts on the same held-out questions.
Ouro uses eight recurrences; Huginn, the retrofits, and Parcae use 32.
Retrofitted models use the checkpoints trained at 32 recurrences. Each matched
control runs once. These counts can be changed in the parameter cell.

Save per-depth accuracies, trained probe weights, tokenizer details, sample
indices, and resolved settings. Residual caches are released after each batch;
plotting uses only the compact accuracy results.

For every model with a coda, also evaluate the coda lens on ARC-Easy. Reuse
the ordinary logit lens's residual cache and send each option's complete,
unpadded sequence through the trained coda. No additional backbone forward pass
or probe training is needed. Observed coda states are omitted from this readout
to avoid applying the coda twice.

For the main-figure Parcae model and its control, the same ARC-Easy cache also
gives a projected logit lens, which applies Parcae's learned projection C before
the ordinary readout.

In [ ]:
examples = {}
sampling = {}
for dataset in datasets:
    test_split = "test" if dataset.startswith("arc_") else "validation"
    train = load_mcq_dataset(dataset, "train")
    test = load_mcq_dataset(dataset, test_split)
    train_indices = sample_indices(
        len(train), min(train_limit, len(train)) if train_limit else None, seed
    )
    test_indices = sample_indices(
        len(test), min(config.limit, len(test)) if config.limit else None, seed
    )
    examples[dataset] = (
        [train[index] for index in train_indices],
        [test[index] for index in test_indices],
    )
    sampling[dataset] = dict(
        train_split="train",
        test_split=test_split,
        train_indices=train_indices,
        test_indices=test_indices,
        n_train=len(train_indices),
        n_test=len(test_indices),
    )
    print(f"{dataset}: {len(train_indices)} train, {len(test_indices)} test")

In [ ]:
settings = asdict(config)
for field in (
    "models",
    "num_steps",
    "task",
    "include_baselines",
    "compute_residual_entropy",
):
    del settings[field]
settings.update(
    dataset="benchmarks",
    output_dir=str(config.output_dir),
    model_recurrences=model_recurrences,
    figure_pairs=figure_pairs,
    datasets=datasets,
    coda_model_recurrences=coda_model_recurrences,
    coda_dataset="arc_easy",
    coda_states="prelude_and_core",
    projected_model_recurrences=projected_pairs,
    projected_dataset="arc_easy",
    projected_states="prelude_and_core",
    max_train=train_limit,
    probe_batch_sizes=probe_batch_sizes,
    probe_learning_rate=probe_learning_rate,
    probe_epochs=1,
    sampling=sampling,
)
settings["versions"] = {
    name: version(name) for name in ("torch", "transformer-lens", "transformers")
}
settings["cache"] = result_cache.settings
run_dir = save_results({}, settings, config.output_dir)

results = {}
rows = []
for pair, pair_config in configs.items():
    name, recurrences = pair
    with ModelSession(name, pair_config, result_cache) as session:
        for dataset in datasets:
            train, test = examples[dataset]
            n_classes = 2 if dataset == "winogrande" else 4
            compute_coda = pair in coda_model_recurrences and dataset == "arc_easy"
            project_pre_coda = pair in projected_pairs and dataset == "arc_easy"
            result, probes, width = collect_benchmark(
                session,
                pair_config,
                result_cache,
                dataset,
                train,
                test,
                batch_size=probe_batch_sizes[name],
                learning_rate=probe_learning_rate,
                compute_coda=compute_coda,
                project_pre_coda=project_pre_coda,
                analysis_version=analysis_version,
                run_dir=run_dir,
            )
            stem = f"{name}-{recurrences}-steps-{dataset}"
            result.save(run_dir / f"{stem}.pt")
            save_probes(
                run_dir / f"{stem}-probes.pt",
                probes,
                width,
                n_classes,
                len(probes),
            )
            results[pair, dataset] = result
            saved_metrics = [
                *METRICS,
                *(["coda_accuracy"] if compute_coda else []),
                *(["projected_logitlens_accuracy"] if project_pre_coda else []),
            ]
            for metric in saved_metrics:
                for block in result.per_virtual_layer_metrics:
                    accuracy = getattr(block, metric).item()
                    if not np.isfinite(accuracy):
                        continue  # Coda-lens readouts exclude the observed coda states.
                    rows.append(
                        dict(
                            model=name,
                            recurrences=recurrences,
                            dataset=dataset,
                            metric=metric,
                            layer_index=block.layer_index,
                            virtual_layer_index=block.virtual_layer_index,
                            accuracy=accuracy,
                            n_train=result.n_train,
                            n_test=result.n_test,
                        )
                    )
            del probes

with (run_dir / "accuracies.csv").open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=rows[0])
    writer.writeheader()
    writer.writerows(rows)
result_cache.write_manifest(run_dir)

## Reading the accuracy curves

| Quantity | Computation |
| --- | --- |
| Linear probe | Mean-pool non-padding tokens in each question-plus-option input, apply a non-affine LayerNorm and a learned scalar head, then choose the highest-scoring option |
| Probe training | A separate head at every virtual layer; one seeded pass with Adam and cross-entropy over options; model weights stay frozen |
| Logit lens | Apply the model's final normalization and unembedding at each virtual layer; choose the option with the highest **summed** continuation log-probability |
| Projected logit lens (Parcae) | Apply Parcae's learned core-output projection C to each prelude/core state, then the ordinary logit lens; coda states are read as usual |
| Coda lens | Apply the trained coda to each prelude/core state, then the final normalization and unembedding; choose the option with the highest summed continuation log-probability |
| Accuracy | Fraction of held-out questions answered correctly at that depth |
| Virtual layer depth (%) | Execution index divided by the largest execution index, scaled to 0–100 |
| Layer depth (%) | Physical layer index divided by the largest physical layer index, scaled to 0–100; repeated applications share an x-coordinate |

WinoGrande options are complete sentences with the blank filled in; its logit
lens scores each sentence using the checkpoint's native BOS convention. The other
tasks score answer continuations. These figures use unnormalised likelihoods.

Green-to-yellow lines show early-to-late recurrences. Prelude and coda are orange
and black; their locations come from recorded layer identities. Each curve is
already an accuracy over the selected questions. There is no averaging across
probe seeds and no uncertainty band. Quick runs check the workflow; their small
training and test samples do not estimate benchmark performance.

The coda readout uses each architecture's native suffix: Huginn normalizes its
raw state before the coda; the retrofits enter the coda directly. Parcae core
states pass through its learned output projection C before the coda; prelude
states already use the coda's residual space and enter directly. Its coda receives
the original input IDs for the trained value embeddings. Attention uses
the original sequence positions and the complete causal context. This decodes an
intermediate state through the trained output layers; it does not resume further
recurrences. Coda accuracies are saved in the per-depth results and CSV, with
unmeasured coda positions represented by NaN in the tensors and omitted from the
CSV and curves. Ouro has no coda, so its panel is marked “No coda”.

Parcae's recurrent state lives in its own space: native execution decodes it only
after C maps the final recurrence into the coda. The projected logit lens applies
that map, without the coda blocks, at every prelude and core point of Parcae 770M
and its feedforward control. For prelude states this is a hypothetical readout,
since natively they enter the recurrence through the injection adapter instead.
An **asterisk on the panel title** marks this convention. The other models have no
such projection and use the ordinary logit lens in both summaries.

In [ ]:
def draw_recurrences(ax, result, metric):
    curves = accuracy_curves(result, metric, by_recurrence=True)
    n_recurrences = sum(group.startswith("R") for group in curves)
    cmap = plt.colormaps[COLORMAPS["recurrences"]]
    lines = {}
    for group, (depth, accuracy) in curves.items():
        if not np.isfinite(accuracy).any():
            continue
        if group == "Prelude":
            color, label = "orange", "Prelude"
        elif group == "Coda":
            color, label = "black", "Coda"
        else:
            color, label = cmap(int(group[1:]) / max(n_recurrences - 1, 1)), None
        (lines[group],) = ax.plot(
            depth, accuracy, color=color, label=label, linewidth=2
        )
    add_prelude_coda_connectors(ax, lines)


def plot_datasets(pairs, filename, *, by_recurrence=False, legend_above=False):
    fig, axes = plt.subplots(
        len(METRICS),
        len(datasets),
        squeeze=False,
        sharex=True,
        sharey="row",
        figsize=get_figsize(
            True, (len(METRICS), len(datasets)), height_per_subplot=1.3
        ),
        layout="constrained",
    )
    for row, (metric, ylabel) in enumerate(METRICS.items()):
        for column, dataset in enumerate(datasets):
            ax = axes[row, column]
            for pair in pairs:
                result = results[pair, dataset]
                if by_recurrence:
                    draw_recurrences(ax, result, metric)
                else:
                    depth, accuracy = accuracy_curves(result, metric)["all"]
                    ax.plot(depth, accuracy, label=MODEL_LABELS[pair[0]])
            ax.set_title(DATASET_LABELS[dataset])
            if row == len(METRICS) - 1:
                ax.set_xlabel(
                    "Layer depth (%)" if by_recurrence else "Virtual layer depth (%)"
                )
            if column == 0:
                ax.set_ylabel(ylabel)
    handles, labels = axes[0, -1].get_legend_handles_labels()
    if handles and legend_above:
        entries = list(zip(handles, labels, strict=True))
        standard = [entry for entry in entries if not entry[1].startswith("Parcae ")]
        parcae = [entry for entry in entries if entry[1].startswith("Parcae ")]
        # Figure legends fill columns first: interleave the two groups so the
        # standard controls occupy the first row and Parcae the second.
        ordered = []
        for index in range(max(len(standard), len(parcae))):
            if index < len(standard):
                ordered.append(standard[index])
            if index < len(parcae):
                ordered.append(parcae[index])
        legend_handles, legend_labels = zip(*ordered, strict=True)
        fig.legend(
            legend_handles,
            legend_labels,
            loc="outside upper center",
            ncols=len(standard),
            fontsize=6,
            frameon=False,
        )
    elif handles:
        axes[0, -1].legend(fontsize=4, loc="lower right")
    if by_recurrence:
        name, recurrences = pairs[0]
        fig.suptitle(f"{MODEL_LABELS[name]} ({recurrences} steps)")
        add_depth_colorbar(fig, axes, "recurrence")
    save_figure(fig, run_dir, filename, bbox_inches=None)


def summary_metric(pair, metric, project_parcae):
    if project_parcae and pair in projected_pairs:
        return "projected_logitlens_accuracy"
    return metric


def plot_summary(pairs, metric, filename, *, dataset="arc_easy", project_parcae=False):
    fig, axes = model_subplots(len(pairs), height_per_subplot=1.8)
    ylabel = "Coda lens accuracy" if metric == "coda_accuracy" else METRICS[metric]
    for index, (ax, pair) in enumerate(zip(axes, pairs, strict=True)):
        projected = project_parcae and pair in projected_pairs
        ax.set(
            title=MODEL_LABELS[pair[0]] + ("*" if projected else ""),
            xlabel="Layer depth (%)",
            xlim=(0, 100),
            ylim=(0, 1),
        )
        ax.locator_params(axis="x", nbins=3)
        if index % 4 == 0:
            ax.set_ylabel(ylabel)
        else:
            ax.tick_params(labelleft=False)
        if metric == "coda_accuracy" and pair not in coda_model_recurrences:
            ax.text(
                0.5, 0.5, "No coda", ha="center", va="center", transform=ax.transAxes
            )
            continue
        baseline_pair = (BASELINE_FOR[pair[0]], 1)
        baseline_metric = summary_metric(
            baseline_pair,
            "logitlens_accuracy" if metric == "coda_accuracy" else metric,
            project_parcae,
        )
        depth, accuracy = accuracy_curves(
            results[baseline_pair, dataset], baseline_metric
        )["all"]
        ax.plot(depth, accuracy, label="Control", linestyle="--", color="red")
        draw_recurrences(
            ax, results[pair, dataset], summary_metric(pair, metric, project_parcae)
        )
    add_prelude_coda_legend(fig, include_coda=metric != "coda_accuracy")
    add_depth_colorbar(fig, axes, "recurrence")
    save_figure(fig, run_dir, filename, bbox_inches=None)

## Feedforward baselines across datasets

A full-width figure compares the standard feedforward models and the 370M
and 770M Parcae feedforward controls at relative execution depth. Rows
show probe and logit-lens accuracy; columns show the four datasets.

In [ ]:
plot_datasets(
    feedforward_figure_pairs, "feedforward_comparison", legend_above=True
)

## Recurrent models across datasets

One full-width figure per model overlays recurrences at their physical layer
positions, retaining separate prelude and coda segments where present.

In [ ]:
for pair in recurrent_pairs:
    plot_datasets([pair], f"{PAPER_NAMES[pair[0]]}_recurrence", by_recurrence=True)

## ARC-Easy: main paper summaries

Separate full-width probe and logit-lens figures show the four main models in
paper order. Each panel overlays its matched feedforward control in dashed red,
using relative execution depth. The shared Prelude/Coda/Control legend appears
above the titles. This compares positions within the models rather than equal
amounts of accumulated computation.

The second logit-lens figure is identical except that Parcae 770M and its
control use the projected logit lens (asterisked titles).

In [ ]:
plot_summary(figure_pairs["main"], "probe_accuracy", "linear_probe_summary")

In [ ]:
plot_summary(figure_pairs["main"], "logitlens_accuracy", "logitlens_summary")

In [ ]:
plot_summary(
    figure_pairs["main"],
    "logitlens_accuracy",
    "logitlens_projected_summary",
    project_parcae=True,
)

## ARC-Easy: accuracy through the trained coda

All eight panels follow the shared paper order, with the main models above the
appendix models. Parcae 770M is in the main row; Parcae 140M and 370M
are in the appendix. Ouro's panel is marked “No coda”. Every other panel overlays its
matched feedforward model's ordinary logit-lens accuracy in dashed red.
Prelude/core curves retain their physical-depth coordinates; observed coda states
are omitted to avoid applying the coda twice.

In [ ]:
plot_summary(figure_pairs["all"], "coda_accuracy", "coda_lens_summary")

In [ ]:
print(run_dir)
paper_dir = export_paper_figures(config, run_dir)
if paper_dir is not None:
    print(f"Saved paper figures to {paper_dir}")